# Phase 1 Worksheet — Document Processing
Read `concept_notes.md` and `diagrams.md` first. Each format below gets a minimal working extraction example, then everything funnels into one ChromaDB collection at the end so you see the full thread from raw file to stored document.

**Corrected in this version:** embeddings now go through `embedder.embed_documents()` (the fixed wrapper) instead of `get_embedding(text, model=MODEL_JINA)`, which doesn't match your real `inhouse_llm.py` signature.

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))  # folder containing the corrected inhouse_wrappers.py
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))  # folder containing your real inhouse_llm.py

from inhouse_llm import MODEL_QWEN3_14B, MODEL_QWEN3_30B, MODEL_MISTRAL, MODEL_LLAMA, MODEL_DEVSTRAL, MODEL_QWEN2_5_VL_7B
from inhouse_wrappers import get_chat_model, InHouseEmbeddings, build_vision_messages, llm_for
from langchain_core.messages import SystemMessage, HumanMessage

embedder = InHouseEmbeddings()

def ask(system_prompt, user_prompt, model=MODEL_QWEN3_14B, max_tokens=200):
    """Drop-in replacement for the old multimodal_chat() text-only calls --
    correctly routed per-model via get_chat_model(), unlike inhouse_llm.py's
    own chat()/multimodal_chat() which always hit the Qwen3-14B endpoint."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]).content

def ask_vision(system_prompt, user_prompt, image_base64, model=MODEL_QWEN2_5_VL_7B, max_tokens=200):
    """Drop-in replacement for multimodal_chat() WITH an image -- uses the
    corrected image_url content-block format, and an actual client for the
    vision model (inhouse_llm.py never created one)."""
    llm = get_chat_model(model=model, max_tokens=max_tokens)
    return llm.invoke(build_vision_messages(system_prompt, user_prompt, image_base64)).content

import chromadb
client = chromadb.HttpClient(host="localhost", port=8000)  # adjust to your Chroma server
print("Setup OK")

## PDF — pypdf vs pdfplumber (this phase's teaser, reproduced)
`pip install pypdf pdfplumber --break-system-packages`. Point `PDF_PATH` at any PDF you have, ideally one with at least one table.

In [ ]:
PDF_PATH = "sample.pdf"  # <-- point at a real file

from pypdf import PdfReader
reader = PdfReader(PDF_PATH)
print("pypdf, page 1 text (no table structure):")
print(reader.pages[0].extract_text()[:500])

import pdfplumber
with pdfplumber.open(PDF_PATH) as pdf:
    page = pdf.pages[0]
    print("\npdfplumber, page 1 text:")
    print(page.extract_text()[:500])
    tables = page.extract_tables()
    print(f"\npdfplumber found {len(tables)} table(s) on page 1 (pypdf has no equivalent)")

## DOCX

In [ ]:
# pip install python-docx --break-system-packages
from docx import Document as DocxDocument

# doc = DocxDocument("sample.docx")
# for para in doc.paragraphs:
#     print(para.style.name, "-", para.text[:80])
# for table in doc.tables:
#     for row in table.rows:
#         print([cell.text for cell in row.cells])
print("Uncomment above and point at a real .docx file.")

## PPT

In [ ]:
# pip install python-pptx --break-system-packages
from pptx import Presentation

# prs = Presentation("sample.pptx")
# for i, slide in enumerate(prs.slides):
#     texts = [shape.text for shape in slide.shapes if shape.has_text_frame]
#     print(f"Slide {i+1}:", " | ".join(texts)[:100])
print("Uncomment above and point at a real .pptx file.")

## HTML

In [ ]:
from bs4 import BeautifulSoup

html = """
<html><body>
<h1>Customer API</h1>
<p>This API manages customer records.</p>
<h2>Authentication</h2>
<p>Requires an API key in the Authorization header.</p>
</body></html>
"""
soup = BeautifulSoup(html, "lxml")
for heading in soup.find_all(["h1", "h2"]):
    print(heading.name, ":", heading.get_text())
print("\nAll paragraph text:", [p.get_text() for p in soup.find_all("p")])

## Markdown, JSON, XML, YAML

In [ ]:
import json, yaml
from lxml import etree

md_text = "# Customer API\n\n## Authentication\nRequires an API key.\n"
print("Markdown headers found:", [line for line in md_text.splitlines() if line.startswith("#")])

json_text = '{"service": "Payment", "endpoints": [{"path": "/pay", "method": "POST"}]}'
parsed_json = json.loads(json_text)
print("\nJSON parsed:", parsed_json)

xml_text = "<service><name>Payment</name><endpoint path='/pay' method='POST'/></service>"
root = etree.fromstring(xml_text)
print("\nXML service name:", root.find("name").text)
print("XML endpoint path (XPath):", root.xpath("//endpoint/@path"))

yaml_text = "service: Payment\nendpoints:\n  - path: /pay\n    method: POST\n"
parsed_yaml = yaml.safe_load(yaml_text)
print("\nYAML parsed:", parsed_yaml)

## CSV, Excel

In [ ]:
import pandas as pd
import io

csv_text = "id,service,status\n1,Payment,Failed\n2,Auth,OK\n"
df_csv = pd.read_csv(io.StringIO(csv_text))
print(df_csv)

# df_excel = pd.read_excel("sample.xlsx", sheet_name=None)  # dict of all sheets
# print({name: sheet.shape for name, sheet in df_excel.items()})
print("\nFor Excel: uncomment above and point at a real .xlsx file.")

## OpenAPI/Swagger

In [ ]:
swagger_text = """
openapi: 3.0.0
paths:
  /payments:
    get:
      summary: List payments
    post:
      summary: Create a payment
  /payments/{id}:
    get:
      summary: Get a payment by id
"""
spec = yaml.safe_load(swagger_text)
for path, methods in spec["paths"].items():
    for method, details in methods.items():
        print(f"{method.upper()} {path}: {details['summary']}")

## Funnel everything into one Chroma collection
Minimal chunking here on purpose (full treatment is Phase 2) — this just closes the loop from "extracted text" to "stored in Chroma". **Corrected:** uses `embedder.embed_documents()` instead of a per-item `get_embedding(t, model=MODEL_JINA)` loop.

In [ ]:
collection = client.get_or_create_collection("phase1_demo")

docs = [
    ("html_auth", "Requires an API key in the Authorization header.", {"source": "html", "section": "Authentication"}),
    ("swagger_get_payments", "GET /payments: List payments", {"source": "swagger", "path": "/payments"}),
    ("csv_row_1", "Transaction 1: service=Payment, status=Failed", {"source": "csv", "id": "1"}),
]
ids = [d[0] for d in docs]
texts = [d[1] for d in docs]
metadatas = [d[2] for d in docs]
embeddings = embedder.embed_documents(texts)

collection.upsert(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
print("Collection count:", collection.count())

## Teaser exercise
Take one real PDF with at least one table. Extract it BOTH with `pypdf` (text only) and `pdfplumber` (text + table). Store both versions as separate Chroma documents with metadata `{"extractor": "pypdf"}` vs `{"extractor": "pdfplumber"}`. Query something that depends on the table's content — does the pdfplumber version retrieve more usefully?